In [12]:
import os
import re
import csv
import json
import pandas as pd
from pathlib import Path
from typing import Optional, Literal
from pydantic import BaseModel, Field
from src.schema import CopyVariant, GenerationBatch, JudgeEvaluation

# Load csv

In [13]:
REQUIRED_FIELDS = ["sku", "title", "orig_price", "promo_price"]

def normalize_and_validate(row: dict) -> tuple[Optional[dict], list[str]]:
    """对单行 CSV 数据进行标准化和规则校验"""
    clean = {k.strip(): (v or "").strip() for k, v in row.items()}
    issues = [f"missing_{f}" for f in REQUIRED_FIELDS if not clean.get(f)]
    
    if issues:
        return None, issues

    try:
        orig = float(clean["orig_price"])
        promo = float(clean["promo_price"])
        if promo > orig:
            issues.append("promo_price_exceeds_original")
    except ValueError:
        issues.append("invalid_price_format")

    if issues:
        return None, issues
    return clean, []

# 读取并展示 promotions.csv
csv_path = "data/promotions.csv" if os.path.exists("data/promotions.csv") else "promotions.csv"
df = pd.read_csv(csv_path)
df.head()

,sku,title,promo_start,promo_end,orig_price,promo_price,category,client,description,size,max_copy_chars
0,SKU_TECH_001,TechNet Dual-Band Router,2026/11/01,11-15-2026,199.99,149.99,Electronics,TechCorp,High-speed wireless router with dual-band support,NaN,200
1,SKU_FASH_001,Summer Breeze Dress,11/05/2026,2026.11.20,49.99,29.99,Apparel,FashionCo,Summer floral dress in breathable cotton,NaN,220
2,SKU_BEAUTY_001,YouthRevive Night Cream,"Nov 10, 2026",25-Nov-26,89.00,75.00,Beauty,FashionCo,Anti-aging night cream enriched with hyaluroni...,50 ml,220
3,SKU_BEVERAGE_001,MorningKick Cold Brew,11/11/26,2026-11-12,12.00,8.00,Groceries,TechCorp,Organic cold brew coffee ready to drink,16 oz,200
4,SKU_HOME_001,GreenEarth Laundry Detergent,2026-11-20 00:00:00,2026-11-30T23:59:59Z,29.99,19.99,Home,FashionCo,Eco-friendly liquid laundry detergent,2 L,220


# Run LLM

In [14]:
def mock_generate(row: dict) -> GenerationBatch:
    """Mock 生成器，用于无需 API 离线调试"""
    client = row.get("client", "")
    title = row.get("title", "")
    orig = row.get("orig_price", "0")
    promo = row.get("promo_price", "0")
    
    if client == "TechCorp":
        simp = f"Save on **${promo}** {title}. High performance! Was ${orig}."
        comp = f"Upgrade your setup with {title}. Save big now at **${promo}** (Was ${orig}). Limited time offer!"
    else:
        simp = f"Summer style ✨ Now ${promo}!"
        comp = f"Discover the gorgeous {title} ✨ Comfortable and breathable. Now ${promo}! Was ${orig}."
        
    variant = CopyVariant(simplified=simp, complicated=comp)
    return GenerationBatch(example_1=variant, example_2=variant, example_3=variant)



# Verify

In [15]:
def rule_checks(copy_text: str, row: dict, client: str) -> dict:
    """第一层确定性规则校验"""
    result = {}
    max_chars = int(row.get('max_copy_chars', 200))
    
    # 1. 基础通用判断
    result['length_ok'] = len(copy_text) <= max_chars
    result['char_count'] = len(copy_text)
    result['price_mentioned'] = any(c in copy_text.lower() for c in ['$', 'price', 'save'])
    
    # 2. 检测 Emoji
    emoji_pattern = r'[\U0001F300-\U0001FAFF]'
    has_emoji = bool(re.search(emoji_pattern, copy_text))
    
    # 3. 客户专属规则判断
    if client == "TechCorp":
        result['price_bolded'] = '**$' in copy_text       # 价格必须加粗
        result['no_emoji'] = not has_emoji               # 禁止 Emoji
        required_keys = ['length_ok', 'price_mentioned', 'price_bolded', 'no_emoji']
    elif client == "FashionCo":
        result['has_emoji'] = has_emoji                  # 必须有 Emoji
        result['price_has_exclamation'] = re.search(r'\$[\d.]+\!', copy_text) is not None  # 价格后带感叹号
        required_keys = ['length_ok', 'price_mentioned', 'has_emoji', 'price_has_exclamation']
    else:
        required_keys = ['length_ok', 'price_mentioned']
        
    result['all_rules_pass'] = all(result.get(k, False) for k in required_keys)
    result['failed_rules'] = [k for k in required_keys if not result.get(k, False)]
    return result

print("规则检查函数定义完成！")

规则检查函数定义完成！


In [ ]:
def mock_llm_judge(copy_text: str, row: dict) -> JudgeEvaluation:
    """Mock LLM 评估返回"""
    return JudgeEvaluation(
        clarity_score=5,
        brand_alignment_score=4,
        overall_pass=True,
        reasoning="符合品牌调性与长度约束。"
    )

print("裁判模块构建完成！")

# Pipeline

In [ ]:
results = []

with open(csv_path, encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

for i, row in enumerate(rows, start=1):
    product, input_issues = normalize_and_validate(row)
    if not product:
        results.append({
            "sku": row.get("sku", f"ROW_{i}"),
            "status": "❌ INVALID_INPUT",
            "issues": input_issues,
            "details": None
        })
        continue

    # 执行文案生成
    batch = mock_generate(product)
    print(batch)
    examples = [batch.example_1, batch.example_2, batch.example_3]
    
    for idx, ex in enumerate(examples, start=1):
        for copy_type, copy_text in [("simplified", ex.simplified), ("complicated", ex.complicated)]:
            # Layer 1: 规则校验
            rule_res = rule_checks(copy_text, product, product.get("client", ""))
            
            # Layer 2: LLM Judge 评估
            judge_res = mock_llm_judge(copy_text, product)
            
            # 综合判断（注意：此处正确使用了 judge_res.overall_pass 属性访问）
            final_pass = rule_res["all_rules_pass"] and judge_res.overall_pass
            
            results.append({
                "sku": product["sku"],
                "client": product.get("client", "N/A"),
                "example_num": idx,
                "copy_type": copy_type,
                "copy_text": copy_text,
                "rule_pass": rule_res["all_rules_pass"],
                "judge_pass": judge_res.overall_pass,
                "status": "✅ PASS" if final_pass else "❌ FAIL",
                "failed_rules": rule_res["failed_rules"],
                "judge_reasoning": judge_res.reasoning,
                "orig_price": product["orig_price"],
                "promo_price": product["promo_price"],
                "title": product["title"]
            })

results_df = pd.DataFrame(results)
print("Pipeline 运行完毕！运行结果预览：")
results_df[["sku", "client", "copy_type", "status", "rule_pass", "judge_pass", "copy_text"]].head(10)

Pipeline 运行完毕！运行结果预览：


,sku,client,copy_type,status,rule_pass,judge_pass,copy_text
0,SKU_TECH_001,TechCorp,simplified,✅ PASS,True,True,Save on **$149.99** TechNet Dual-Band Router. ...
1,SKU_TECH_001,TechCorp,complicated,✅ PASS,True,True,Upgrade your setup with TechNet Dual-Band Rout...
2,SKU_TECH_001,TechCorp,simplified,✅ PASS,True,True,Save on **$149.99** TechNet Dual-Band Router. ...
3,SKU_TECH_001,TechCorp,complicated,✅ PASS,True,True,Upgrade your setup with TechNet Dual-Band Rout...
4,SKU_TECH_001,TechCorp,simplified,✅ PASS,True,True,Save on **$149.99** TechNet Dual-Band Router. ...
5,SKU_TECH_001,TechCorp,complicated,✅ PASS,True,True,Upgrade your setup with TechNet Dual-Band Rout...
6,SKU_FASH_001,FashionCo,simplified,❌ FAIL,False,True,Summer style ✨ Now $29.99!
7,SKU_FASH_001,FashionCo,complicated,❌ FAIL,False,True,Discover the gorgeous Summer Breeze Dress ✨ Co...
8,SKU_FASH_001,FashionCo,simplified,❌ FAIL,False,True,Summer style ✨ Now $29.99!
9,SKU_FASH_001,FashionCo,complicated,❌ FAIL,False,True,Discover the gorgeous Summer Breeze Dress ✨ Co...


In [8]:
TILE_HTML = """
<div style="border:1px solid #e0e0e0;border-radius:8px;padding:14px;width:220px;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif;box-shadow:0 2px 4px rgba(0,0,0,0.05);background:#fff">
  <span style="font-size:10px;font-weight:bold;background:#f0f0f0;color:#555;padding:2px 6px;border-radius:4px">{client}</span>
  <h4 style="margin:8px 0 4px;font-size:14px;color:#333">{title}</h4>
  <p style="font-size:12px;color:#666;margin:0 0 10px;line-height:1.4">{copy_text}</p>
  <div style="color:#d32f2f;font-weight:bold;font-size:16px">
    ${promo_price} <span style="color:#999;font-weight:normal;font-size:12px;text-decoration:line-through">${orig_price}</span>
  </div>
</div>
"""

os.makedirs("outputs", exist_ok=True)
valid_cards = [TILE_HTML.format(**r) for r in results if r["status"] == "✅ PASS"]

html_content = f"""
<!DOCTYPE html>
<html>
<head><meta charset="utf-8"><title>Flyer Preview</title></head>
<body style="background:#f9f9f9;padding:20px">
  <h2 style="font-family:sans-serif;color:#333">Promotional Copy Flyer Preview</h2>
  <div style="display:flex;gap:16px;flex-wrap:wrap">
    {"".join(valid_cards)}
  </div>
</body>
</html>
"""

html_path = Path("outputs/flyer_preview.html")
html_path.write_text(html_content, encoding="utf-8")
print(f"HTML 预览已成功生成：{html_path.resolve()}")

HTML 预览已成功生成：/Users/Test/coding-projects/data-processing/outputs/flyer_preview.html


In [9]:
from IPython.display import HTML

# 读取生成的 HTML 文件并直接在 Notebook 中渲染输出
with open("outputs/flyer_preview.html", "r", encoding="utf-8") as f:
    html_content = f.read()

# 显示 HTML
HTML(html_content)